In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
import pandas as pd
import numpy as np
import joblib
import warnings

In [ ]:
warnings.filterwarnings("ignore")
X = pd.read_parquet("data/processed/X.parquet")
Y = pd.read_parquet("data/processed/Y.parquet")

split = int(len(X) * 0.85)
X_train, Y_train = X.iloc[:split], Y.iloc[:split]
X_test, Y_test = X.iloc[split:], Y.iloc[split:]

In [ ]:
numeric_features = [c for c in X_train.columns if c not in ["month", "day_of_week"]]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), ["day_of_week", "month"]),
])

ridge = Pipeline([
    ("prep", preprocess),
    ("reg", Ridge()), 
])

In [ ]:
ridge_search = GridSearchCV(
    ridge,
    {"reg__alpha": [0.01, 0.05, 0.1, 0.5,  1, 5, 10, 50, 75, 100, 175, 250, 500, 1000]},
    cv=TimeSeriesSplit(n_splits=10),
    scoring="neg_mean_squared_error",
    n_jobs=-1,
)

ridge_search.fit(X_train, Y_train)

print(ridge_search.best_params_)
print((-ridge_search.best_score_) ** 0.5)

best_ridge = ridge_search.best_estimator_

In [ ]:
pred = best_ridge.predict(X_test)
err = pred - Y_test.values
rmse = np.sqrt(np.mean(err**2))
print(f"Ridge holdout RMSE: {rmse:.2f}")
print(f"Ridge holdout MAE: {np.mean(np.abs(err)):.2f}")

In [ ]:
# re-fitting Ridge regression with best alpha on all available data

best_ridge = Pipeline([
    ("prep", preprocess),
    ("reg", Ridge(alpha=50)), 
])

best_ridge.fit(X, Y)
joblib.dump(best_ridge, "models/ridge.joblib")